# Project 1: Build a Calculator from Scratch

**Level**: Beginner  
**Skills covered**: functions, loops, conditionals, exception handling, lists, f-strings

---

## Project Goal

You will build a fully functional calculator step by step — starting with simple arithmetic functions and ending with a robust, interactive calculator that keeps a history of calculations and supports scientific operations.

By the end you will have:
- A reusable set of arithmetic functions with proper error handling
- An interactive REPL loop that accepts user input
- A calculation history stored in a list
- Input validation that never crashes on bad input
- Pretty formatted output
- Bonus: scientific calculator extensions

---

## Steps Overview

1. Basic arithmetic functions
2. Main calculator loop
3. Adding history
4. Input validation
5. Pretty output formatting
6. Bonus: scientific operations

---

## Step 1: Basic Arithmetic Functions

Start with four simple functions — one per operation. The only tricky case is division by zero.

In [ ]:
# --- Naive first version ---

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    return a / b   # will crash if b == 0

# Quick test
print(add(10, 5))       # 15
print(subtract(10, 5))  # 5
print(multiply(10, 5))  # 50
print(divide(10, 5))    # 2.0

In [ ]:
# --- Improved version: handle division by zero ---

def add(a, b):
    """Return the sum of a and b."""
    return a + b

def subtract(a, b):
    """Return a minus b."""
    return a - b

def multiply(a, b):
    """Return a times b."""
    return a * b

def divide(a, b):
    """Return a divided by b. Raises ValueError on division by zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b

# Test division by zero
try:
    print(divide(10, 0))
except ValueError as e:
    print(f"Caught error: {e}")

print(divide(10, 4))  # 2.5

**Key concepts**: raising exceptions with `raise`, catching them with `try/except`, docstrings.

---

## Step 2: Main Calculator Function

Wire the four functions together with a `while` loop and conditional branching.

In [ ]:
# --- Naive version: no validation, no history ---

def calculator_v1():
    print("Simple Calculator (type 'quit' to exit)")
    while True:
        expression = input("Enter calculation (e.g. 10 + 5): ")
        if expression.lower() == "quit":
            break

        parts = expression.split()
        a, operator, b = float(parts[0]), parts[1], float(parts[2])

        if operator == "+":
            result = add(a, b)
        elif operator == "-":
            result = subtract(a, b)
        elif operator == "*":
            result = multiply(a, b)
        elif operator == "/":
            result = divide(a, b)
        else:
            print("Unknown operator")
            continue

        print(f"Result: {result}")

# Uncomment to run interactively:
# calculator_v1()

In [ ]:
# Demonstrate the core logic without blocking input

OPERATIONS = {
    "+": add,
    "-": subtract,
    "*": multiply,
    "/": divide,
}

def calculate(a, operator, b):
    """Dispatch to the right function based on operator string."""
    if operator not in OPERATIONS:
        raise ValueError(f"Unknown operator: '{operator}'")
    return OPERATIONS[operator](a, b)

# Test the dispatcher
for op in ["+", "-", "*", "/"]:
    print(f"10 {op} 4 = {calculate(10, op, 4)}")

**Key concepts**: dictionaries as dispatch tables, `while True` / `break`, `continue`.

---

## Step 3: Adding Calculation History

Store each calculation in a list so the user can review past results.

In [ ]:
# History is just a list of strings (or dicts for richer data)

history = []

def record(a, operator, b, result):
    """Append a calculation record to history."""
    entry = {
        "expression": f"{a} {operator} {b}",
        "result": result,
    }
    history.append(entry)

def show_history():
    """Print all past calculations."""
    if not history:
        print("No calculations yet.")
        return
    print("\n--- Calculation History ---")
    for i, entry in enumerate(history, start=1):
        print(f"  {i}. {entry['expression']} = {entry['result']}")
    print("---------------------------\n")

# Simulate a few calculations
record(10, "+", 5, calculate(10, "+", 5))
record(20, "/", 4, calculate(20, "/", 4))
record(7, "*", 3, calculate(7, "*", 3))

show_history()

**Key concepts**: list of dicts, `enumerate`, f-string formatting, functions that modify outer-scope lists.

---

## Step 4: Input Validation

Real users type unexpected things. Wrap all input parsing in `try/except` so the calculator never crashes.

In [ ]:
def parse_input(raw):
    """
    Parse a string like '10 + 5' into (float, str, float).
    Raises ValueError with a helpful message on bad input.
    """
    parts = raw.strip().split()

    if len(parts) != 3:
        raise ValueError(
            f"Expected format: <number> <operator> <number>, got {len(parts)} parts."
        )

    raw_a, operator, raw_b = parts

    try:
        a = float(raw_a)
    except ValueError:
        raise ValueError(f"'{raw_a}' is not a valid number.")

    try:
        b = float(raw_b)
    except ValueError:
        raise ValueError(f"'{raw_b}' is not a valid number.")

    if operator not in OPERATIONS:
        raise ValueError(f"'{operator}' is not a supported operator. Use: {list(OPERATIONS)}.")

    return a, operator, b


# Test validation
test_inputs = [
    "10 + 5",        # valid
    "abc + 5",       # bad first number
    "10 % 5",        # bad operator
    "10 / 0",        # valid parse, but divide will raise
    "10 + 5 + 3",    # too many parts
]

for expr in test_inputs:
    try:
        a, op, b = parse_input(expr)
        result = calculate(a, op, b)
        print(f"  OK  | {expr} = {result}")
    except ValueError as e:
        print(f" ERR  | {expr!r} -> {e}")

**Key concepts**: defensive parsing, re-raising exceptions with a better message, `repr()` via `!r` in f-strings.

---

## Step 5: Pretty Output Formatting

Make output clean — align columns, show integers without `.0`, limit decimal places.

In [ ]:
def format_number(n):
    """Return an integer string if n is whole, else 6 significant figures."""
    if n == int(n):
        return str(int(n))
    return f"{n:.6g}"


def format_result(a, operator, b, result):
    """Return a nicely formatted result line."""
    fa, fb, fr = format_number(a), format_number(b), format_number(result)
    return f"  {fa:>12} {operator} {fb:<12} =  {fr}"


# Demo
cases = [
    (10, "+", 5),
    (1000000, "*", 999),
    (1, "/", 3),
    (22, "/", 7),
    (100, "-", 0.001),
]

print("Calculation Results")
print("-" * 40)
for a, op, b in cases:
    try:
        result = calculate(a, op, b)
        print(format_result(a, op, b, result))
    except ValueError as e:
        print(f"  Error: {e}")

**Key concepts**: f-string alignment (`:<`, `:>`), `:.6g` format spec, conditional formatting.

---

## Step 6: Putting It All Together — Full Calculator

Combine all steps into a single `Calculator` class with a `run()` method.

In [ ]:
import math

class Calculator:
    """Interactive calculator with history, validation, and pretty output."""

    BASIC_OPS = {
        "+": add,
        "-": subtract,
        "*": multiply,
        "/": divide,
        "%": lambda a, b: a % b,
        "**": lambda a, b: a ** b,
    }

    def __init__(self):
        self.history = []

    # --- arithmetic ---

    def calculate(self, a, operator, b):
        if operator not in self.BASIC_OPS:
            raise ValueError(f"Unknown operator '{operator}'. Supported: {list(self.BASIC_OPS)}")
        result = self.BASIC_OPS[operator](a, b)
        self.history.append({"expression": f"{a} {operator} {b}", "result": result})
        return result

    # --- parsing ---

    def parse(self, raw):
        parts = raw.strip().split()
        if len(parts) != 3:
            raise ValueError("Format: <number> <operator> <number>")
        raw_a, operator, raw_b = parts
        try:
            a = float(raw_a)
        except ValueError:
            raise ValueError(f"'{raw_a}' is not a number")
        try:
            b = float(raw_b)
        except ValueError:
            raise ValueError(f"'{raw_b}' is not a number")
        return a, operator, b

    # --- display ---

    @staticmethod
    def fmt(n):
        return str(int(n)) if n == int(n) else f"{n:.6g}"

    def show_history(self):
        if not self.history:
            print("No history yet.")
            return
        print("\n--- History ---")
        for i, e in enumerate(self.history, 1):
            print(f"  {i:3}. {e['expression']} = {self.fmt(e['result'])}")
        print("---------------\n")

    # --- main loop ---

    def run(self):
        print("Calculator  |  operators: + - * / % **  |  commands: history, clear, quit")
        while True:
            try:
                raw = input(">>> ").strip()
            except (KeyboardInterrupt, EOFError):
                print("\nGoodbye!")
                break

            if not raw:
                continue
            if raw.lower() == "quit":
                print("Goodbye!")
                break
            if raw.lower() == "history":
                self.show_history()
                continue
            if raw.lower() == "clear":
                self.history.clear()
                print("History cleared.")
                continue

            try:
                a, op, b = self.parse(raw)
                result = self.calculate(a, op, b)
                print(f"  = {self.fmt(result)}")
            except ValueError as e:
                print(f"  Error: {e}")


# Uncomment to run:
# calc = Calculator()
# calc.run()

print("Calculator class defined. Uncomment the last two lines to run interactively.")

---

## Bonus: Scientific Calculator

Extend the calculator with `sqrt`, `log`, `sin`, `cos`, and `factorial` using the `math` module.

In [ ]:
import math

SCIENTIFIC_OPS = {
    "sqrt": math.sqrt,
    "log": math.log10,     # log base 10
    "ln": math.log,        # natural log
    "sin": math.sin,       # radians
    "cos": math.cos,
    "tan": math.tan,
    "abs": abs,
    "factorial": math.factorial,
}

def scientific_calc(func_name, value):
    """
    Apply a scientific function to a single value.
    Returns (result, error_message).
    """
    if func_name not in SCIENTIFIC_OPS:
        return None, f"Unknown function '{func_name}'. Available: {list(SCIENTIFIC_OPS)}"
    try:
        result = SCIENTIFIC_OPS[func_name](value)
        return result, None
    except (ValueError, OverflowError) as e:
        return None, str(e)


# Demo all scientific functions
test_cases = [
    ("sqrt", 144),
    ("log", 1000),
    ("ln", math.e),
    ("sin", math.pi / 2),
    ("cos", 0),
    ("factorial", 10),
    ("sqrt", -1),     # error case
]

print(f"{'Function':<12} {'Input':>10}    {'Result'}")
print("-" * 40)
for fn, val in test_cases:
    result, err = scientific_calc(fn, val)
    if err:
        print(f"{fn:<12} {val:>10}  ->  ERROR: {err}")
    else:
        print(f"{fn:<12} {val:>10}  ->  {result:.6g}")

---

## Summary

What you built and the skills you practiced:

| Step | What you built | Skills |
|------|---------------|--------|
| 1 | Arithmetic functions | functions, exceptions |
| 2 | Dispatch loop | while loop, dict dispatch, conditionals |
| 3 | History | list of dicts, enumerate |
| 4 | Input validation | try/except, re-raising, defensive parsing |
| 5 | Pretty output | f-string format specs |
| 6 | OOP refactor | class, `__init__`, static methods |
| Bonus | Scientific ops | math module, function references |

---

## How to Extend This

- **Persistent history**: save `history` to a JSON file on exit with `json.dump()` and reload on start
- **Expression parser**: parse full expressions like `(3 + 4) * 2` using Python's `ast.literal_eval` or the `sympy` library
- **Unit conversion**: add operators like `->km`, `->mph` to convert between units
- **GUI**: wrap the `Calculator` class in a `tkinter` window with buttons
- **Web app**: serve it as a Flask API endpoint that evaluates posted expressions
- **Fraction mode**: use `fractions.Fraction` to keep results exact (e.g., `1/3` stays `1/3`)